# 1. Explain your own classifier with SCE, end to end

This is the README section *"How to no-code use a custom binary image classification dataset with a
predictor and a generator from PEAL"* as a notebook.

You start with a folder of images and a binary label. You end with counterfactual explanations of a
classifier PEAL trained for you. Everything is driven by config files that are copies of the shipped
CelebA ones with a few keys changed.

**Cost warning.** Two cells train models from scratch. The DDPM generator is the expensive one and
takes GPU-days on a real dataset; the classifier takes minutes to hours. Run them on a machine with
a GPU, or point the configs at checkpoints you already have.

In [ ]:
import os, subprocess, sys, textwrap
from pathlib import Path

# PEAL resolves <PEAL_BASE> itself; $PEAL_DATA and $PEAL_RUNS default to ./datasets and ./peal_runs.
PEAL_BASE = Path(os.environ.get("PEAL_BASE", Path.cwd().parent)).resolve()
PEAL_DATA = Path(os.environ.get("PEAL_DATA", PEAL_BASE / "datasets"))
PEAL_RUNS = Path(os.environ.get("PEAL_RUNS", PEAL_BASE / "peal_runs"))
os.chdir(PEAL_BASE)          # every entry point expects the repo root as the working directory
os.environ["PEAL_DATA"] = str(PEAL_DATA)
os.environ["PEAL_RUNS"] = str(PEAL_RUNS)
print("PEAL_BASE", PEAL_BASE, "\nPEAL_DATA", PEAL_DATA, "\nPEAL_RUNS", PEAL_RUNS)

def run(cmd):
    """Stream a PEAL command. These are long jobs; the output is the run log."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    code = p.wait()
    if code != 0:
        # Without this a failed generator training is followed by a classifier training and an
        # SCE run that cannot work, and the first symptom is a missing .npz six cells later.
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return code


In [ ]:
import yaml

def patch_yaml(template, target, **changes):
    """Copy a shipped config and overwrite single top-level keys.

    Comments are lost (yaml round-trips values only), which is why the README tells you to copy the
    files by hand. Doing it programmatically keeps a notebook reproducible; open the template if you
    want to read what each field means.
    """
    src = PEAL_BASE / template
    dst = PEAL_BASE / target
    dst.parent.mkdir(parents=True, exist_ok=True)
    cfg = yaml.safe_load(src.read_text())
    for k, v in changes.items():
        cfg[k] = v
    dst.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(f"wrote {target}")
    return dst


## Step 1 — put the dataset where PEAL reads it

PEAL's `Image2MixedDataset` wants `$PEAL_DATA/my_data/imgs/...` plus a `data.csv` whose first column
is the path of each image relative to `imgs`, followed by one column per label.

Point `SOURCE_DIR` at your images. The cell below writes the csv; adapt `label_of()` to your own
labelling rule.

In [ ]:
import csv
DATASET = "my_data"          # <- name your dataset
SOURCE_DIR = PEAL_DATA / DATASET / "imgs"

def label_of(path):
    """Return 0 or 1 for an image path. Replace with your own rule."""
    return 1 if path.parent.name.lower().startswith("positive") else 0

rows = [(str(p.relative_to(SOURCE_DIR)), label_of(p))
        for p in sorted(SOURCE_DIR.rglob("*"))
        if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
csv_path = PEAL_DATA / DATASET / "data.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["imgs", "Label1"]); w.writerows(rows)
print(f"{len(rows)} images -> {csv_path}")
print("class balance:", {c: sum(1 for r in rows if r[1] == c) for c in (0, 1)})


## Step 2 — derive the configs

Five configs, all copies of the CelebA ones with a handful of keys replaced. Do not change
`input_size` unless you know the generator supports it: the generative model is fixed in that
regard.

In [ ]:
data_cfg = patch_yaml(
    "configs/sce_experiments/data/celeba.yaml",
    f"configs/my_experiments/data/{DATASET}.yaml",
    dataset_path=f"$PEAL_DATA/{DATASET}",
    # celeba.yaml declares CelebA's 40 attribute columns; your data.csv has one label column.
    # Leave output_type as multiclass: dataset_factory uses it to pick Image2MixedDataset once
    # dataset_class is popped below.
    output_size=[1],
)
gen_data_cfg = patch_yaml(
    "configs/sce_experiments/data/celeba_generator.yaml",
    f"configs/my_experiments/data/{DATASET}_generator.yaml",
    dataset_path=f"$PEAL_DATA/{DATASET}",
)
# the README tells you to drop these two, because you have neither for a fresh dataset
for p in (data_cfg, gen_data_cfg):
    cfg = yaml.safe_load(p.read_text())
    cfg.pop("dataset_class", None); cfg.pop("confounding_factors", None)
    p.write_text(yaml.safe_dump(cfg, sort_keys=False))

patch_yaml("configs/sce_experiments/generators/celeba_ddpm.yaml",
           f"configs/my_experiments/generators/{DATASET}_ddpm.yaml",
           base_path=f"$PEAL_RUNS/{DATASET}/ddpm",
           data=f"<PEAL_BASE>/configs/my_experiments/data/{DATASET}_generator.yaml")

patch_yaml("configs/sce_experiments/predictors/celeba_Smiling_classifier.yaml",
           f"configs/my_experiments/predictors/{DATASET}_classifier.yaml",
           model_path=f"$PEAL_RUNS/{DATASET}/classifier",
           data=f"<PEAL_BASE>/configs/my_experiments/data/{DATASET}.yaml",
           task={"criterions": {"ce": 1.0}, "output_type": "singleclass",
                 "output_channels": 2, "y_selection": ["Label1"]})

patch_yaml("configs/sce_experiments/adaptors/celeba_Smiling_natural_sce_cfkd.yaml",
           f"configs/my_experiments/adaptors/{DATASET}_sce_cfkd.yaml",
           data=f"<PEAL_BASE>/configs/my_experiments/data/{DATASET}.yaml",
           student=f"$PEAL_RUNS/{DATASET}/classifier/model.cpl",
           generator=f"$PEAL_RUNS/{DATASET}/ddpm/config.yaml",
           base_dir=f"$PEAL_RUNS/{DATASET}/classifier/sce_cfkd",
           calculate_explainer_stats=False,
           task={"criterions": {"ce": 1.0}, "output_type": "singleclass",
                 "output_channels": 2, "y_selection": ["Label1"]})


## Step 3 — train the generator  *(the long one)*

In [ ]:
run(f'python train_generator.py --config "<PEAL_BASE>/configs/my_experiments/generators/{DATASET}_ddpm.yaml"')

## Step 4 — train the classifier you want explained

In [ ]:
run(f'python train_predictor.py --config "<PEAL_BASE>/configs/my_experiments/predictors/{DATASET}_classifier.yaml"')

## Step 5 — run SCE

Counterfactuals land in `validation_collages0_1` (the most salient ones) and `validation_collages0_0`
(secondary; these can look destroyed when SCE had to force a second, different counterfactual).

In [ ]:
run(f'python run_cfkd.py --config "<PEAL_BASE>/configs/my_experiments/adaptors/{DATASET}_sce_cfkd.yaml"')

## Step 6 — look at what came out

The raw arrays are in `validation_tracked_values.npz`: `x_list` holds the factuals and
`x_counterfactual_list` the counterfactuals, in matching order.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def show_counterfactuals(npz_path, n=4, title=""):
    """Plot factual/counterfactual pairs out of a run's tracked values."""
    z = np.load(npz_path, allow_pickle=True)
    x  = np.asarray(z["x_list"], dtype=np.float32)
    cf = np.asarray(z["x_counterfactual_list"], dtype=np.float32)
    n = min(n, len(cf))
    fig, ax = plt.subplots(n, 2, figsize=(4, 2 * n))
    ax = np.atleast_2d(ax)
    img = lambda a: np.clip(a.transpose(1, 2, 0), 0, 1)
    for i in range(n):
        ax[i, 0].imshow(img(x[i])); ax[i, 0].set_ylabel(f"#{i}", fontsize=8)
        ax[i, 1].imshow(img(cf[i]))
        for j in (0, 1):
            ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        if i == 0:
            ax[i, 0].set_title("factual", fontsize=9)
            ax[i, 1].set_title("counterfactual", fontsize=9)
    fig.suptitle(title); fig.tight_layout(); plt.show()


In [ ]:
npz = PEAL_RUNS / DATASET / "classifier" / "sce_cfkd" / "0" / "validation_tracked_values.npz"
print("exists:", npz.exists())
if npz.exists():
    show_counterfactuals(npz, n=4, title="SCE counterfactuals")


## Where to go next

* To *correct* the classifier rather than only explain it, configure a CFKD adaptor; the shipped
  examples are under `configs/cfkd_experiments/adaptors`.
* If you already have a trained model and do not want PEAL to train one, see notebook 2, which takes
  an ONNX file instead.
* If you want to explain a foundation-model probe without training any generator at all, see
  notebook 3.